## Automated Hyperparameter Selection 

This tutorial demonstrates how to automatically select optimal Gaussian Process hyperparameters for your surrogate model by testing multiple configurations and choosing the one with the best test set performance.

### Why This Matters

GP surrogate model performance is highly sensitive to:
- **Kernel choice** (ExpSquared, Matern32, Matern52, etc.)
- **Data scaling** (no scaling, MinMax, StandardScaler)
- **Other hyperparameters** (white noise, amplitude, length scales)

Rather than manually tuning these, we can systematically test combinations and select the best configuration based on test set mean squared error (MSE).

### Step 1: Import Required Libraries

Import `alabi` and its submodules along with standard scientific computing libraries.

In [1]:
import alabi
import alabi.utility as ut
import alabi.benchmarks as bm
from alabi.core import SurrogateModel

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import preprocessing
from itertools import product

np.random.seed(101)

### Step 2: Define Problem and Base Configuration

Set up the benchmark problem (eggbox function) and define a base configuration for GP hyperparameters. These settings will be partially overridden when we test different combinations.

**Key parameters:**
- `ninit`: Number of initial training points
- `niter`: Number of active learning iterations
- `ncore`: Number of CPU cores for parallel evaluation (use 1 if experiencing multiprocessing issues)
- `white_noise`: Log-scale white noise parameter (increase if you see positive definiteness errors)

In [2]:
ninit = 50
niter = 100
basedir = "demo"
kernel = "ExpSquaredKernel"
benchmark = "rosenbrock"
savedir = f"{basedir}/{benchmark}/{kernel}/{ninit}_{niter}"

gp_kwargs = {"kernel": kernel, 
             "fit_amp": True, 
             "fit_mean": True, 
             "fit_white_noise": False, 
             "white_noise": -12,
             "gp_opt_method": "l-bfgs-b",
             "hyperopt_method": "cv",
             "cv_folds": 8,
             "gp_amp_rng": [-1,1],
             "gp_scale_rng": [-2,2],
             "theta_scaler": alabi.no_scaler,
             "y_scaler": alabi.no_scaler}
    
sm = SurrogateModel(lnlike_fn=bm.eggbox["fn"], 
                    bounds=bm.eggbox["bounds"], 
                    savedir=savedir,
                    ncore=8, 
                    pool_method="forkserver",
                    verbose=True)

sm.init_samples(ntrain=ninit, ntest=1000, sampler="sobol")

  0%|          | 0/50 [00:00<?, ?it/s]

  2%|▏         | 1/50 [00:01<01:35,  1.94s/it]

100%|██████████| 50/50 [00:02<00:00, 24.93it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 1/1000 [00:01<30:32,  1.83s/it]

  0%|          | 2/1000 [00:02<14:59,  1.11it/s]

100%|██████████| 1000/1000 [00:02<00:00, 478.92it/s]

### Step 3: Create Hyperparameter Grid

Generate all combinations of the hyperparameters we want to test:
- **Kernels**: Different covariance functions capture different smoothness assumptions
- **Theta scaler**: How to scale input parameters
- **Y scaler**: How to scale output values

The `dict_to_combinations` function creates a Cartesian product of all options.

In [3]:
def dict_to_combinations(options_dict):
    keys = options_dict.keys()
    values = options_dict.values()
    return [dict(zip(keys, combo)) for combo in product(*values)]

def combinations_to_dict(combinations):
    result = {key: [] for key in combinations[0].keys()}
    for combo in combinations:
        for key, value in combo.items():
            result[key].append(value)
    return result

gp_kwarg_options = {"kernel": ["ExpSquaredKernel", "Matern32Kernel", "Matern52Kernel"],
                    "theta_scaler": [ut.no_scaler, preprocessing.MinMaxScaler(), preprocessing.StandardScaler()],
                    "y_scaler": [ut.no_scaler, preprocessing.MinMaxScaler(), preprocessing.StandardScaler()]}

variable_settings = dict_to_combinations(gp_kwarg_options)
print(len(variable_settings), "combinations to test")

# get a list of dictionaries with all combinations of settings, where each dictionary is a copy of the original gp_kwargs with the variable settings updated
setting_combos = []
for settings in variable_settings:
    new_settings = gp_kwargs.copy()
    for key in settings.keys():
        new_settings[key] = settings[key]
    setting_combos.append(new_settings)

27 combinations to test


### Step 4: Test All Hyperparameter Combinations

Loop through all combinations and fit a GP for each. The `init_gp` method returns the test set MSE, which we use to evaluate performance.

**Note:** This can take several minutes depending on the number of combinations and problem dimensionality. Use `try/except` to handle configurations that fail to converge.

In [4]:
for ii in range(len(setting_combos)):
    try:
        test_mse = sm.init_gp(**setting_combos[ii], overwrite=True)
    except:
        test_mse = np.nan
    setting_combos[ii]["test_mse"] = test_mse

Initialized GP with squared exponential kernel.
Successfully initialized GP on attempt 1

Optimizing GP hyperparameters using 8-fold cross-validation...


Evaluating 100 hyperparameter candidates using 8-fold CV...
Using multiprocessing pool with 8 processes


Evaluating candidates:   0%|          | 0/100 [00:00<?, ?candidate/s]

Evaluating candidates:   1%|          | 1/100 [00:01<02:53,  1.75s/candidate]

Evaluating candidates:   3%|▎         | 3/100 [00:01<00:48,  1.98candidate/s]

...

Evaluating candidates: 100%|██████████| 100/100 [00:01<00:00, 74.89candidate/s]

Stage 2 candidates:   0%|          | 0/50 [00:00<?, ?candidate/s]

Stage 2 candidates: 100%|██████████| 50/50 [00:00<00:00, 618.06candidate/s]

Stage 3 candidates:   0%|          | 0/25 [00:00<?, ?candidate/s]

Stage 3 candidates: 100%|██████████| 25/25 [00:00<00:00, 615.41candidate/s]

### Step 5: Analyze Results

Convert results to a pandas DataFrame and inspect the top-performing configurations. Lower test MSE indicates better generalization performance.

In [5]:
results = pd.DataFrame(data=combinations_to_dict(setting_combos))
top_fits = results.sort_values("test_mse").head(5)
top_fits

,kernel,fit_amp,fit_mean,fit_white_noise,white_noise,gp_opt_method,hyperopt_method,cv_folds,gp_amp_rng,gp_scale_rng,theta_scaler,y_scaler,test_mse
25,Matern52Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",StandardScaler(),MinMaxScaler(),3014.105741
17,Matern32Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",StandardScaler(),StandardScaler(),3085.244354
13,Matern32Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",MinMaxScaler(),MinMaxScaler(),3262.803131
16,Matern32Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",StandardScaler(),MinMaxScaler(),3274.704553
9,Matern32Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",no_scaler,no_scaler,3300.396040


### Step 6: Extract Best Configuration

Identify the hyperparameter configuration with the lowest test MSE. This will be used for active learning.

In [6]:
best_gp_results = results[results["test_mse"] == results["test_mse"].min()]
best_gp_results

,kernel,fit_amp,fit_mean,fit_white_noise,white_noise,gp_opt_method,hyperopt_method,cv_folds,gp_amp_rng,gp_scale_rng,theta_scaler,y_scaler,test_mse
25,Matern52Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",StandardScaler(),MinMaxScaler(),3014.105741


### Step 7: Run Active Learning

Use the optimal GP configuration for active learning. The BAPE (Bayesian Active Posterior Estimation) algorithm iteratively selects new training points to improve the surrogate model.

**Key active learning parameters:**
- `algorithm`: Acquisition function ("bape", "agp", or "jones")
- `gp_opt_freq`: How often to reoptimize GP hyperparameters during training
- `obj_opt_method`: Optimization method for acquisition function
- `nopt`: Number of optimization restarts for acquisition function

In [7]:
best_gp_kwargs = best_gp_results[gp_kwargs.keys()].to_dict(orient="records")[0]

al_kwargs = {"algorithm": "bape", 
             "gp_opt_freq": 20, 
             "obj_opt_method": "nelder-mead", 
             "nopt": 6}

sm.init_gp(**best_gp_kwargs, overwrite=True)
sm.active_train(niter=200, **al_kwargs)

Initialized GP with Matérn-5/2 kernel.
Successfully initialized GP on attempt 1

Optimizing GP hyperparameters using 8-fold cross-validation...


Evaluating 100 hyperparameter candidates using 8-fold CV...
Using multiprocessing pool with 8 processes


Evaluating candidates:   0%|          | 0/100 [00:00<?, ?candidate/s]

Evaluating candidates:   1%|          | 1/100 [00:01<01:58,  1.20s/candidate]

Evaluating candidates:  68%|██████▊   | 68/100 [00:01<00:00, 72.03candidate/s]

...

Stage 3 candidates:   0%|          | 0/25 [00:00<?, ?candidate/s]

Stage 3 candidates:  92%|█████████▏| 23/25 [00:00<00:00, 217.23candidate/s]

Stage 3 candidates: 100%|██████████| 25/25 [00:00<00:00, 228.87candidate/s]


100%|██████████| 200/200 [00:26<00:00,  7.46it/s]

Train MSE: 0.0731375680913975
Test MSE: 128.48926569367472
Caching model to demo/rosenbrock/ExpSquaredKernel/50_100/surrogate_model...


### Step 8: Visualize Training Progress

Plot the test set MSE over active learning iterations. A decreasing trend indicates the surrogate model is improving. We can also highlight which iterations the GP hyperparameters are re-optimized (vertical gray lines).

In [8]:
plt.plot(sm.training_results["iteration"], sm.training_results["test_mse"])
for ii in range(0, sm.nactive, sm.gp_opt_freq+1):
    plt.axvline(ii, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("Iteration", fontsize=18)
plt.ylabel("Test MSE", fontsize=18)
plt.xlim(0, sm.nactive)
plt.show()

RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./file.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2023/05/17 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))

! LaTeX Error: File `type1cm.sty' not found.

Type X to quit or <RETURN> to proceed,
or enter new name. (Default extension: sty)

Enter file name: 
! Emergency stop.
<read *> 
         
l.7 \usepackage
               {type1ec}^^M
No pages of output.
Transcript written on file.log.




<Figure size 640x480 with 1 Axes>

How do the other top initial fits perform during active learning?

In [9]:
al_kwargs = {"algorithm": "bape", 
             "gp_opt_freq": 20, 
             "obj_opt_method": "nelder-mead", 
             "nopt": 6}

mse_results = {}
for idx in top_fits.index:
    gp_kwargs_idx = top_fits[top_fits.index == idx][gp_kwargs.keys()].to_dict(orient="records")[0]
    sm.init_gp(**gp_kwargs_idx, overwrite=True)
    sm.active_train(niter=200, **al_kwargs)
    mse_results[idx] = sm.training_results["test_mse"]

Initialized GP with Matérn-5/2 kernel.
Successfully initialized GP on attempt 1

Optimizing GP hyperparameters using 8-fold cross-validation...


Evaluating 100 hyperparameter candidates using 8-fold CV...
Using multiprocessing pool with 8 processes


Evaluating candidates:   0%|          | 0/100 [00:00<?, ?candidate/s]

Evaluating candidates:   1%|          | 1/100 [00:01<01:58,  1.20s/candidate]

Evaluating candidates:  92%|█████████▏| 92/100 [00:01<00:00, 97.75candidate/s]

...

Stage 3 candidates:   0%|          | 0/25 [00:00<?, ?candidate/s]

Stage 3 candidates: 100%|██████████| 25/25 [00:00<00:00, 283.97candidate/s]


100%|██████████| 200/200 [00:41<00:00,  1.41it/s]

100%|██████████| 200/200 [00:41<00:00,  4.80it/s]

Train MSE: 1.8418115975059023e-17
Test MSE: 178.92904102359094
Caching model to demo/rosenbrock/ExpSquaredKernel/50_100/surrogate_model...


In [10]:
plt.figure(figsize=(10,6))
for idx in top_fits.index:
    plt.plot(sm.training_results["iteration"], mse_results[idx], label=f"config {idx}")
for ii in range(0, sm.nactive, sm.gp_opt_freq+1):
    plt.axvline(ii, color="gray", linestyle="--", alpha=0.5)
plt.legend(loc="upper right", fontsize=16)
plt.xlabel("Iteration", fontsize=18)
plt.ylabel("Test MSE", fontsize=18)
plt.xlim(0, sm.nactive)
plt.show()

top_fits

RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./file.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2023/05/17 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))

! LaTeX Error: File `type1cm.sty' not found.

Type X to quit or <RETURN> to proceed,
or enter new name. (Default extension: sty)

Enter file name: 
! Emergency stop.
<read *> 
         
l.7 \usepackage
               {type1ec}^^M
No pages of output.
Transcript written on file.log.




<Figure size 1000x600 with 1 Axes>

,kernel,fit_amp,fit_mean,fit_white_noise,white_noise,gp_opt_method,hyperopt_method,cv_folds,gp_amp_rng,gp_scale_rng,theta_scaler,y_scaler,test_mse
25,Matern52Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",StandardScaler(),MinMaxScaler(),3014.105741
17,Matern32Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",StandardScaler(),StandardScaler(),3085.244354
13,Matern32Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",MinMaxScaler(),MinMaxScaler(),3262.803131
16,Matern32Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",StandardScaler(),MinMaxScaler(),3274.704553
9,Matern32Kernel,True,True,False,-12,l-bfgs-b,cv,8,"[-1, 1]","[-2, 2]",no_scaler,no_scaler,3300.396040


### Next Steps

Now that you have an optimized surrogate model, you can:

1. **Run posterior sampling** with `sm.run_emcee()` or `sm.run_dynesty()`
2. **Visualize the surrogate** using `alabi.visualization` tools
3. **Test on new problems** by changing the benchmark function
4. **Expand the hyperparameter grid** to include more options like:
   - Different `white_noise` values
   - Different `cv_folds` settings
   - Different data scaling functions for `theta_scaler` or `y_scaler`